# Sequence interpolation — gh0st_flux_lora_v2

Generates a smooth morph video from the 61 FLUX sequence stills using optical flow.
No Runway clips — continuous blend from rhinestones → bead_cage → crybabyglitch → red_blackliner.

**Before running:** upload the sequence stills folder to Google Drive:
```
Local:  spikes/flux_lora_training/output/gh0st_flux_lora_v2/sequence/
Drive:  Gh0st in the Loop/outputs/gh0st_flux_lora_v2/sequence/
```
The folder structure should be:
```
sequence/
  pure/rhinestones.png, bead_cage.png, crybabyglitch.png, red_blackliner.png
  bead_cage_x_rhinestones/005bead_cage__095rhinestones.png … 095bead_cage__005rhinestones.png
  crybabyglitch_x_bead_cage/…
  red_blackliner_x_crybabyglitch/…
```

In [ ]:
import os

!pip install opencv-python-headless imageio imageio-ffmpeg -q

from google.colab import drive
drive.mount('/content/drive')

if os.path.exists(repo_dir := '/content/gh0st-in-the-l00p'):
    !git -C {repo_dir} pull --quiet
else:
    !git clone --quiet https://github.com/jasonr2048/gh0st-in-the-l00p.git {repo_dir}

%cd {repo_dir}
print('Ready.')

In [ ]:
from datetime import datetime
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT   = Path('/content/drive/MyDrive/Gh0st in the Loop')
SEQUENCE_DIR = DRIVE_ROOT / 'outputs' / 'gh0st_flux_lora_v2' / 'sequence'
OUTPUT_DIR   = DRIVE_ROOT / 'outputs'

# ── Timing ────────────────────────────────────────────────────────────────────
FPS          = 24
HOLD_FRAMES  = 0    # frames to hold each still before morphing (0 = no hold)
MORPH_FRAMES = 16   # frames of optical flow morph between consecutive stills
                    # 16 frames @ 24fps = ~0.67s per step; 60 steps → ~40s total

# ── Output ────────────────────────────────────────────────────────────────────
# Source stills are 1024×1024 square. Crop to portrait for exhibition screens,
# or keep square. Set OUTPUT_SIZE = (1024, 1024) to keep square.
OUTPUT_SIZE  = (1024, 1024)   # (width, height)

timestamp    = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_PATH  = OUTPUT_DIR / f'sequence_interp_{timestamp}.mp4'
print(f'Output: {OUTPUT_PATH}')

In [ ]:
# Build ordered list of stills — same sequence order as assemble_video.py

ALL_ALPHAS = [round(v / 100, 2) for v in range(5, 100, 5)]  # 0.05 … 0.95

def pure(style: str) -> Path:
    return SEQUENCE_DIR / 'pure' / f'{style}.png'

def transition(set_a: str, set_b: str) -> list[Path]:
    """Clips in ascending alpha (set_b → set_a direction)."""
    pair_dir = SEQUENCE_DIR / f'{set_a}_x_{set_b}'
    paths = []
    for alpha in ALL_ALPHAS:
        pct_a = int(round(alpha * 100))
        pct_b = 100 - pct_a
        p = pair_dir / f'{pct_a:03d}{set_a}__{pct_b:03d}{set_b}.png'
        if p.exists():
            paths.append(p)
        else:
            print(f'  WARNING: missing {p.name}')
    return paths

stills = [
    pure('rhinestones'),
    *transition('bead_cage', 'rhinestones'),
    pure('bead_cage'),
    *transition('crybabyglitch', 'bead_cage'),
    pure('crybabyglitch'),
    *transition('red_blackliner', 'crybabyglitch'),
    pure('red_blackliner'),
]

missing = [p for p in stills if not p.exists()]
print(f'Stills: {len(stills)} total, {len(missing)} missing')
if missing:
    for m in missing:
        print(f'  MISSING: {m}')
    stills = [p for p in stills if p.exists()]
    print(f'Proceeding with {len(stills)} stills.')

expected_duration_s = (len(stills) * HOLD_FRAMES + (len(stills) - 1) * MORPH_FRAMES) / FPS
print(f'Expected duration: {expected_duration_s:.1f}s ({expected_duration_s / 60:.1f} min) at {FPS}fps')

In [ ]:
# Preview: show every 6th still to check the sequence looks right
from PIL import Image
import IPython.display as ipd

sample = stills[::6]
thumbs = [Image.open(p).convert('RGB').resize((160, 160)) for p in sample]
grid = Image.new('RGB', (len(thumbs) * 160, 160))
for i, t in enumerate(thumbs):
    grid.paste(t, (i * 160, 0))
ipd.display(grid)
print('Every 6th still: ' + ' | '.join(p.stem[:20] for p in sample))

In [ ]:
import cv2
import numpy as np
from PIL import Image

def load(path: Path, size: tuple) -> np.ndarray:
    img = Image.open(path).convert('RGB')
    w, h = size
    # Centre-crop to target aspect ratio, then resize
    src_w, src_h = img.size
    tgt_ratio = w / h
    src_ratio = src_w / src_h
    if abs(src_ratio - tgt_ratio) > 0.01:
        if src_ratio > tgt_ratio:
            new_w = int(src_h * tgt_ratio)
            left = (src_w - new_w) // 2
            img = img.crop((left, 0, left + new_w, src_h))
        else:
            new_h = int(src_w / tgt_ratio)
            top = (src_h - new_h) // 2
            img = img.crop((0, top, src_w, top + new_h))
    return np.array(img.resize((w, h), Image.LANCZOS))

def optical_flow_morph(a: np.ndarray, b: np.ndarray, steps: int) -> list[np.ndarray]:
    a_gray = cv2.cvtColor(a, cv2.COLOR_RGB2GRAY)
    b_gray = cv2.cvtColor(b, cv2.COLOR_RGB2GRAY)
    flow = cv2.calcOpticalFlowFarneback(
        a_gray, b_gray, None,
        pyr_scale=0.5, levels=3, winsize=15,
        iterations=3, poly_n=5, poly_sigma=1.2, flags=0
    )
    h, w = a.shape[:2]
    xs = np.tile(np.arange(w), (h, 1)).astype(np.float32)
    ys = np.tile(np.arange(h), (w, 1)).T.astype(np.float32)
    frames = []
    for t in np.linspace(0, 1, steps, endpoint=False):
        wx = (xs + flow[..., 0] * t).astype(np.float32)
        wy = (ys + flow[..., 1] * t).astype(np.float32)
        warped = cv2.remap(a, wx, wy, cv2.INTER_LINEAR)
        blended = cv2.addWeighted(warped, 1 - t, b, t, 0)
        frames.append(blended)
    return frames

print('Functions defined.')

In [ ]:
import imageio

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load all stills upfront (61 × 1024×1024 RGB ≈ 180 MB — fine for Colab)
print(f'Loading {len(stills)} stills...')
frames_loaded = [load(p, OUTPUT_SIZE) for p in stills]
print('Done loading.')

all_frames = []
for i, img in enumerate(frames_loaded):
    if HOLD_FRAMES > 0:
        all_frames.extend([img] * HOLD_FRAMES)
    if i < len(frames_loaded) - 1:
        all_frames.extend(optical_flow_morph(img, frames_loaded[i + 1], MORPH_FRAMES))
    if i % 10 == 0:
        print(f'  {i}/{len(frames_loaded)} stills processed ({len(all_frames)} frames so far)')

# Add hold on final frame
if HOLD_FRAMES > 0:
    all_frames.extend([frames_loaded[-1]] * HOLD_FRAMES)

total_s = len(all_frames) / FPS
print(f'\nRendering {len(all_frames)} frames ({total_s:.1f}s) → {OUTPUT_PATH.name}')

with imageio.get_writer(str(OUTPUT_PATH), fps=FPS) as writer:
    for i, frame in enumerate(all_frames):
        writer.append_data(frame)
        if i % 100 == 0:
            print(f'  {i}/{len(all_frames)} frames written...', end='\r')

print(f'\n✅ Saved to {OUTPUT_PATH}')

In [ ]:
import json

sidecar = {
    'experiment_id': OUTPUT_PATH.stem,
    'duration_seconds': round(total_s, 3),
    'fps': FPS,
    'source': 'flux_lora_v2_sequence_stills',
    'n_stills': len(stills),
    'hold_frames': HOLD_FRAMES,
    'morph_frames': MORPH_FRAMES,
    'output_size': OUTPUT_SIZE,
    'generated_at': datetime.now().isoformat(),
}
sidecar_path = OUTPUT_PATH.with_suffix('.json')
sidecar_path.write_text(json.dumps(sidecar, indent=2))
print(f'✅ Sidecar written to {sidecar_path.name}')
print(json.dumps(sidecar, indent=2))

In [ ]:
from IPython.display import HTML
from base64 import b64encode

data_url = 'data:video/mp4;base64,' + b64encode(open(OUTPUT_PATH, 'rb').read()).decode()
display(HTML(f'<video width=540 controls autoplay loop><source src="{data_url}" type="video/mp4"></video>'))